Final models to run on the dataset

Run on python 3.13.13 kernel, on Visual Studios Code, using Juypter notebook.
 4_FINAL_prepared_dataset.csv is used in this file

Installs

In [ ]:
%pip install torch torchvision torchaudio
%pip install transformers
%pip install transformers datasets torch

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
   ---------------------------------------- 0.0/645.5 kB ? eta -:--:--
   ---------------------------------------- 645.5/645.5 kB 10.7 MB/s  0:00:00
Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl (3.7 MB)

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface-hub 0.36.0

    Uninstalling huggingface-hub-0.36.0:

      Successfully uninstalled huggingface-hub-0.36.0

   ------------- -------------------------- 1/3 [huggingface-hub]
   ------------- -------------------------- 1/3 [huggingface-hub]
   ------------- -------------------------- 1/3 [huggingface-hub]
   ------------- -------------------------- 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.5.4 which is incompatible.


Defaulting to user installation because normal site-packages is not writeable


Installing alternative versions if issues occur prior

In [ ]:
%python -m pip uninstall -y transformers
%python -m pip install "transformers>=4.41.0,<5.0.0"
%python -m pip install "accelerate>=0.26.0"
%python -m pip install --upgrade datasets

Found existing installation: transformers 5.5.4
Uninstalling transformers-5.5.4:
  Successfully uninstalled transformers-5.5.4
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   -------------------- ------------------- 6.3/12.0 MB 47.2 MB/s eta 0:00:01
   ---------------------------------------  11.8/12.0 MB 40.0 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 24.5 MB/s  0:00:00
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB 3.2 MB/s  0:00:00

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 1.11.0

    Uninstalling huggingface_hub-1.11.0:

      Successfully uninstalled huggingface_hub-1.11.0

   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   -------------

In [ ]:
#checking the versions of install
#default install were to modern
import sys
print(sys.executable)

import transformers
print(transformers.__version__)
import accelerate
print(accelerate.__version__)

imports and setting label map based on what combination is used

In [1]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

# DistilBERT specifics
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from torch.nn import CrossEntropyLoss

# load the dataset
df = pd.read_csv("4_FINAL_prepared_dataset_N.csv")
df = df[["label", "clean_text"]]

#CHANGE BASED ON WHAT DATASET YOU ARE USING
label_map = {
    "AI_phish": 0,
    "phishing": 1
}
#setting text based labels to numerical labels
df["label_encoded"] = df["label"].map(label_map)

X = df["clean_text"]
y = df["label_encoded"]


In [2]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Compute weights from only training data
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
#convert the weights to dictionary as this format is used by sklearn models
class_weight_dict = dict(zip(classes, class_weights))
print("Class weights:", class_weight_dict)

# For XGBoost binary classification:
# labels are 0 = AI, 1 = phishing or benign
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

# 1 Classic Models
#configuring the models
models = {
    #ensures convergence, class weighting and keeping reproducibility
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42, class_weight=class_weight_dict),
    "Naive Bayes": MultinomialNB(),
    #class weighting and reproducibility
    "Linear SVM": LinearSVC(random_state=42, class_weight=class_weight_dict),
    #200 trees, class weighting and reproducibility
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight=class_weight_dict),
    "XGBoost": XGBClassifier(
        n_estimators=200, #number of tree
        max_depth=6, # tree depth
        learning_rate=0.1, # step size
        subsample=0.8, # randomness
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight # XGBoost's weighting
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        #Apply TF-IDF vectorization to the data
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000
        )),
        ("model", model)
    ])
    #pipeline on the train test split
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    #getting results
    acc = accuracy_score(y_test, y_pred)
    results.append((name, acc))

    #printing results
    print(name)
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))

# 2 DistilBERT

# Compute class weights from y_train

#get classes
classes = np.unique(y_train)
class_weights = compute_class_weight( # compute class weights
    class_weight="balanced",
    classes=classes,
    y=y_train
)
#convert to dictionary
class_weight_dict = dict(zip(classes, class_weights))
print("Class weights:", class_weight_dict)
#converting to PyTorch tensor, enforces it to pay attention to rare class
weight_tensor = torch.tensor(
    [class_weight_dict[0], class_weight_dict[1]],
    dtype=torch.float
)

#create training DF
train_df = pd.DataFrame({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})
#create test DF
test_df = pd.DataFrame({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

#loading the variant of BERT and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#configuring the tokenizer
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )
#tokenizing the training and test datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)
#remove raw text as its no longer needed
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
#convert dataset outputs into PyTorch tensors
train_dataset.set_format("torch")
test_dataset.set_format("torch")

#load the distilBERT
distilBERT_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
#defining the evaluation metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

#custom trainer with weighted loss to account for imbalance
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        #weighted cross-entropy loss function
        loss_fct = CrossEntropyLoss(weight=weight_tensor.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

#defining the training arguments, controlling how its run
training_args = TrainingArguments(
    output_dir="./distilbert_results", # folder of checkpoints
    eval_strategy="epoch", #evaluate at end of epoch
    save_strategy="epoch", # save checkpoint after epoch
    logging_dir="./distilbert_logs", # directory for logs
    per_device_train_batch_size=8, # train with 8 examples per batch
    per_device_eval_batch_size=8, # same for evaluation
    num_train_epochs=3,# train for 3 passes through the training set
    weight_decay=0.01, # adding regularization to reduce overfitting
    load_best_model_at_end=True, # reload best checkpoint
    metric_for_best_model="accuracy", # use accuracy to decide best checkpoint
    save_total_limit=1, # save only the best to conserve disk space
    report_to="none" # disable external logging integrations
)

#create the trainer
trainer = WeightedTrainer(
    model=distilBERT_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
#start the trainer
trainer.train()

# Predict on test set
predictions = trainer.predict(test_dataset)
distilBERT_preds = np.argmax(predictions.predictions, axis=-1)

distilBERT_acc = accuracy_score(y_test, distilBERT_preds)
results.append(("DistilBERT", distilBERT_acc))

#print scores
print("DistilBERT")
print(f"Accuracy: {distilBERT_acc:.4f}")
print(classification_report(y_test, distilBERT_preds))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, distilBERT_preds))

# 3 Summary
results_df = pd.DataFrame(results, columns=["Model", "Accuracy"]).sort_values(
    by="Accuracy", ascending=False
)

print("\nModel Comparison:")
print(results_df.to_string(index=False))

Class weights: {np.int64(0): np.float64(1.273125), np.int64(1): np.float64(0.8233629749393695)}
scale_pos_weight: 0.6467259498787389
Logistic Regression
Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       310

    accuracy                           1.00       510
   macro avg       1.00      1.00      1.00       510
weighted avg       1.00      1.00      1.00       510


Confusion matrix:
[[200   0]
 [  0 310]]
Naive Bayes
Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       310

    accuracy                           1.00       510
   macro avg       1.00      1.00      1.00       510
weighted avg       1.00      1.00      1.00       510


Confusion matrix:
[[200   0]
 [  0 310]]
Linear SVM
Accuracy: 1.0000
              precision    recall  

Map:   0%|          | 0/2037 [00:00<?, ? examples/s]

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\jwboy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.000140,1.000000
2,0.010700,0.000051,1.000000
3,0.010700,0.000038,1.000000


C:\Users\jwboy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\jwboy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\jwboy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DistilBERT
Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       310

    accuracy                           1.00       510
   macro avg       1.00      1.00      1.00       510
weighted avg       1.00      1.00      1.00       510


Confusion matrix:
[[200   0]
 [  0 310]]

Model Comparison:
              Model  Accuracy
Logistic Regression       1.0
        Naive Bayes       1.0
         Linear SVM       1.0
      Random Forest       1.0
            XGBoost       1.0
         DistilBERT       1.0
